# ECGR 4106 Homework 1 — CIFAR-10 CNN Comparison

**Student:** Andrew Dominguez Luna  
**Course:** ECGR 4106 Deep Learning  
**Homework:** Homework 1  
**Framework:** PyTorch  

This notebook is meant to be opened **after the GitHub repository has already been downloaded or cloned**.

The TA should be able to open this notebook from the project folder and run the cells to test the code.

The project compares:

- Modified AlexNet
- Adapted VGGNet
- ResNet-11
- ResNet-18

## Expected folder layout

This notebook expects the following files and folders to be in the **same project folder** as the notebook:

```text
README.md
requirements.txt
train.py
quick_model_test.py
plot_summary.py
models/
utils/
results/
report/
```

If this notebook cannot find `train.py`, `requirements.txt`, `models/`, and `utils/`, then the repository was probably uploaded with an extra nested folder. In that case, move the project files to the repository root.

## 0. Optional Colab GPU setup

If running in Google Colab, enable GPU first:

`Runtime → Change runtime type → Hardware accelerator → T4 GPU`

The training code automatically uses CUDA if it is available. Otherwise, it uses CPU.

## 1. Confirm current folder

This cell prints the current working directory and lists the files. The goal is to confirm that the notebook is running from the root of the homework repository.

In [ ]:
from pathlib import Path

print("Current folder:")
print(Path.cwd())

print("\nFiles in current folder:")
!ls

## 2. Verify repository files

This cell checks that the required project files exist. If this fails, the notebook is not being run from the correct folder.

In [ ]:
from pathlib import Path

required_paths = [
    "requirements.txt",
    "train.py",
    "quick_model_test.py",
    "plot_summary.py",
    "models",
    "utils",
]

missing = [p for p in required_paths if not Path(p).exists()]

if missing:
    print("Missing required files/folders:")
    for p in missing:
        print(" -", p)
    raise FileNotFoundError("Notebook is not running from the project root folder.")
else:
    print("All required project files were found.")

## 3. Install requirements

This installs the packages needed for the homework. In Colab, many packages may already be installed, so seeing `Requirement already satisfied` is normal.

In [ ]:
!pip install -r requirements.txt

## 4. Check PyTorch and hardware

This confirms whether PyTorch can see a GPU. The homework can run on CPU, but GPU is recommended because there are many training runs.

In [ ]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## 5. Dataset explanation

This project uses CIFAR-10, which contains 60,000 color images across 10 classes:

- airplane
- automobile
- bird
- cat
- deer
- dog
- frog
- horse
- ship
- truck

The code uses this split:

| Split | Images |
|---|---:|
| Training | 45,000 |
| Validation | 5,000 |
| Test | 10,000 |

CIFAR-10 is downloaded automatically by `torchvision`. The dataset is not stored in GitHub.

## 6. Quick model test

This checks that all model classes can accept CIFAR-10-shaped input and produce 10 output logits. It does not train the models.

In [ ]:
!python quick_model_test.py

# Problem 1 — Modified AlexNet on CIFAR-10

AlexNet was originally designed for large ImageNet images. CIFAR-10 images are only `32 x 32`, so the original AlexNet is too large and downsamples too aggressively.

The modified AlexNet used here changes the architecture by:

- using smaller `3 x 3` convolution filters,
- reducing the number of channels,
- reducing the size of the fully connected layers,
- keeping enough spatial resolution for CIFAR-10.

The modified AlexNet has **3,192,458 trainable parameters**, compared with about **61 million** parameters in the original AlexNet.

## Problem 1A — Baseline modified AlexNet

This trains the modified AlexNet with no dropout. For a quick TA test, run 1 epoch. For the final homework report, run 30 epochs.

In [ ]:
# Quick test version:
!python train.py --model alexnet --dropout 0.0 --epochs 1 --lr 0.01

# Final homework version:
# !python train.py --model alexnet --dropout 0.0 --epochs 30 --lr 0.01

## Problem 1B — AlexNet dropout experiments

Dropout is used as a regularization method. It randomly disables neurons during training, which can reduce overfitting.

The assignment asks for at least two dropout rates. This project uses:

- `p = 0.3`
- `p = 0.5`

In [ ]:
# Quick test versions:
!python train.py --model alexnet --dropout 0.3 --epochs 1 --lr 0.01
!python train.py --model alexnet --dropout 0.5 --epochs 1 --lr 0.01

# Final homework versions:
# !python train.py --model alexnet --dropout 0.3 --epochs 30 --lr 0.01
# !python train.py --model alexnet --dropout 0.5 --epochs 30 --lr 0.01

# Problem 2 — Adapted VGGNet on CIFAR-10

VGGNet uses repeated `3 x 3` convolution blocks. Original VGG models are large, so this implementation reduces the channel count and classifier size for CIFAR-10.

The adapted VGG model is chosen so its parameter count is close to the modified AlexNet, which makes the comparison more fair.

In [ ]:
# Quick test versions:
!python train.py --model vgg --dropout 0.0 --epochs 1 --lr 0.01
!python train.py --model vgg --dropout 0.3 --epochs 1 --lr 0.01
!python train.py --model vgg --dropout 0.5 --epochs 1 --lr 0.01

# Final homework versions:
# !python train.py --model vgg --dropout 0.0 --epochs 30 --lr 0.01
# !python train.py --model vgg --dropout 0.3 --epochs 30 --lr 0.01
# !python train.py --model vgg --dropout 0.5 --epochs 30 --lr 0.01

# Problem 3 — ResNet-11 vs ResNet-18 on CIFAR-10

ResNet uses skip connections. A skip connection lets information bypass a block and be added back later. This helps deeper networks train more effectively.

For CIFAR-10, this project uses a CIFAR-style ResNet stem:

- `3 x 3` first convolution,
- no initial `7 x 7` convolution,
- no initial max pooling.

This avoids shrinking the `32 x 32` CIFAR-10 images too early.

In [ ]:
# Quick test versions:
!python train.py --model resnet11 --dropout 0.0 --epochs 1 --lr 0.1
!python train.py --model resnet18 --dropout 0.0 --epochs 1 --lr 0.1

# Final homework versions:
# !python train.py --model resnet11 --dropout 0.0 --epochs 50 --lr 0.1
# !python train.py --model resnet11 --dropout 0.3 --epochs 50 --lr 0.1
# !python train.py --model resnet11 --dropout 0.5 --epochs 50 --lr 0.1
# !python train.py --model resnet18 --dropout 0.0 --epochs 50 --lr 0.1
# !python train.py --model resnet18 --dropout 0.3 --epochs 50 --lr 0.1
# !python train.py --model resnet18 --dropout 0.5 --epochs 50 --lr 0.1

# Result files

Each training run creates a folder inside `results/`.

Example:

```text
results/alexnet_dropout00/
```

Each result folder contains:

- `config.json` — hyperparameters, hardware, seed, and parameter count
- `training_log.csv` — training and validation metrics per epoch
- `final_results.json` — test loss, test accuracy, and best validation accuracy
- `loss_curve.png` — training and validation loss plot
- `val_accuracy_curve.png` — validation accuracy plot
- `confusion_matrix.png` — test confusion matrix
- `best_model.pt` — best model checkpoint

In [ ]:
!find results -maxdepth 2 -type f | sort

# Create final comparison summary

After the full experiments are complete, run this script to create:

- `results/summary_table.csv`
- `results/best_model_test_accuracy_bar_chart.png`

In [ ]:
!python plot_summary.py

# Save results to Google Drive, optional

Colab runtimes are temporary. If running in Colab, save the `results/` folder to Google Drive so the experiment outputs are not lost.

In [ ]:
# Optional, only needed in Google Colab:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r results /content/drive/MyDrive/HW1_CIFAR10_results